# AI Deep Interview Empath

Анализ глубинных-интервью

**Пайплайн:**

1. Загрузка аудио (приведение к 16 kHz моно).
2. **Диаризация** - кто и когда говорит (модель `pyannote/speaker-diarization-3.1`).
3. **Транскрипция** - что именно сказано (`faster-whisper large-v3`).
4. **Роли** - определяем интервьюера и респондента по таймингам.
5. **Q/A-сегментация** - режем поток на пары «вопрос → ответ».
6. **Эмоции** - получаем таймлайн эмоций в интервью.
7. **Финальный результат** - JSON со всех полученной информацией.

## 0. Установка зависимостей и конфиг

In [ ]:
%%capture

%pip uninstall -y numpy
%pip install -q --upgrade --no-cache-dir pyarrow numpy numba librosa pyannote.audio transformers datasets

%cd /content

# import os
# os.kill(os.getpid(), 9)

In [ ]:
!mkdir -p test_interviews && cd test_interviews && \
 wget -q --show-progress -O 01_debian_ep04_1546.mp3 'https://archive.org/download/debianrupodcast/debianpodcast04.mp3' && \
 wget -q --show-progress -O 02_radio_interview_1900.mp3 'https://archive.org/download/interview-2023/2023-01-29_S_dumoj_o_Nashih_detjah.mp3' && \
 wget -q --show-progress -O 03_nba_podcast_2247.mp3 'https://archive.org/download/Mix22m47saudioJoiner.com/mix_22m47s%20%28audio-joiner.com%29.mp3' && \
 cd /content && ls

01_debian_ep04_1546 100%[===================>]  21.79M  65.2MB/s    in 0.3s    
02_radio_interview_ 100%[===================>]  17.47M  31.3MB/s    in 0.6s    
03_nba_podcast_2247 100%[===================>]  52.17M  55.1MB/s    in 0.9s    
02_radio_interview_1900.analysis.json  ser_head_dusha.pt
sample_data			       test_interviews


#### !!! КАКОЕ ИНТЕРВЬЮ ИЗ ТЕСТОВЫХ ОБРАБОТАТЬ? (1-3)

In [ ]:
import os

INTERVIEW_NUM = 2  # 1 - подкаст Debian, 2 - радиоинтервью (результаты ниже), 3 - NBA-подкаст

# Токен Hugging Face с принятым соглашением для pyannote/speaker-diarization-3.1.
# В коде не хранится: берётся из Colab Secrets (HF_TOKEN) или переменной окружения.
try:
    from google.colab import userdata
    HF_TOKEN = userdata.get("HF_TOKEN")
except Exception:
    HF_TOKEN = os.environ.get("HF_TOKEN", "")


In [ ]:
import json
import re
import uuid

from typing import Optional

import torch
import torchaudio
from pydantic import BaseModel
from IPython.display import Audio, display

import os
from pathlib import Path

DIR_PATH = Path("/content/test_interviews")
interviews = sorted(DIR_PATH / f for f in os.listdir(DIR_PATH))  # sorted: порядок listdir не гарантирован

AUDIO_PATH = interviews[INTERVIEW_NUM - 1]

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

print(f"Аудиофайл: {AUDIO_PATH}")
print(f"Устройство: {DEVICE}")

Аудиофайл: /content/test_interviews/02_radio_interview_1900.mp3
Устройство: cuda


## 1. Загрузка аудио

Приводим аудио к 16 kHz моно для единообразия формата для всех этапов пайплайна.

In [ ]:
def load_audio_16k_mono(path: Path) -> tuple[torch.Tensor, int]:
    waveform, sr = torchaudio.load(str(path))
    if waveform.shape[0] > 1:                    # стерео → моно усреднением каналов
        waveform = waveform.mean(dim=0, keepdim=True)
    if sr != 16000:                              # ресемплинг до 16 kHz
        waveform = torchaudio.functional.resample(waveform, sr, 16000)
        sr = 16000
    return waveform, sr


waveform, sr = load_audio_16k_mono(AUDIO_PATH)
duration_sec = waveform.shape[-1] / sr
print(f"Длительность: {duration_sec:.1f} сек, sample rate: {sr} Hz, форма: {tuple(waveform.shape)}")

Audio(str(AUDIO_PATH))

Длительность: 1140.0 сек, sample rate: 16000 Hz, форма: (1, 18239616)


## 2. Структура выходных данных пайплайна

Контейнер — `InterviewAnalysis` — собирает в себя всё:
- метаданные аудио (`AudioMeta`),
- список спикеров (`Speaker[]` с ролями),
- сегменты речи (`Segment[]` с текстом и словами),
- найденные вопросы (`Question[]` со ссылками на сегменты-ответы).

In [ ]:
class Speaker(BaseModel):
    id: str                         # "S0", "S1"
    label_src: Optional[str] = None # как звали в источнике, "SPEAKER_00"
    role: Optional[str] = None      # "interviewer" / "respondent"


class Word(BaseModel):
    w: str
    start: float
    end: float
    conf: float


class Segment(BaseModel):
    seg_id: str
    start: float
    end: float
    speaker: Optional[str] = None
    text: Optional[str] = None
    words: Optional[list[Word]] = None


class Question(BaseModel):
    start: float
    end: float
    text: str
    asked_by: str           # id спикера-интервьюера


class EmotionPoint(BaseModel):
    '''Одна точка сплошного таймлайна эмоций (окно фиксированной длины).'''
    start: float
    end: float
    speaker: str                  # кто доминирует в окне по диаризации
    dominant: str
    scores: dict[str, float]


class AudioMeta(BaseModel):
    title: str
    uri: Optional[str] = None
    duration_sec: float
    sample_rate: int = 16000


class InterviewAnalysis(BaseModel):
    job_id: str
    audio: AudioMeta
    speakers: list[Speaker] = []
    segments: list[Segment] = []
    questions: list[Question] = []
    emotion_timeline: list[EmotionPoint] = []


contract = InterviewAnalysis(
    job_id=str(uuid.uuid4()),
    audio=AudioMeta(
        title=AUDIO_PATH.stem,
        uri=str(AUDIO_PATH),
        duration_sec=duration_sec,
    ),
)

print(contract.model_dump_json(indent=2))

{
  "job_id": "e86f00e6-b054-4680-9549-2cf632ba401e",
  "audio": {
    "title": "02_radio_interview_1900",
    "uri": "/content/test_interviews/02_radio_interview_1900.mp3",
    "duration_sec": 1139.976,
    "sample_rate": 16000
  },
  "speakers": [],
  "segments": [],
  "questions": [],
  "emotion_timeline": []
}


## 3. Диаризация

Используем `pyannote/speaker-diarization-3.1`

На выходе получаем список вида `(start, end, label)`, `label` — абстрактный идентификатор спикера, который модель присвоила сама (`SPEAKER_00`, `SPEAKER_01`, и тд).
Мы переименуем их в `S0`, `S1` для читаемости.

Модель часто рвёт длинную монологичную реплику на 5-10 кусочков по 0.5–2 секунды (микропаузы между словами). Если оставить как есть, дальше анализировать «текст реплики» будет тяжело. Склеиваем подряд идущие turn'ы одного и того же спикера, если пауза между ними меньше `MERGE_GAP_SEC` секунд.

In [ ]:
from pyannote.audio import Pipeline

DIAR_MODEL    = "pyannote/speaker-diarization-3.1"
MERGE_GAP_SEC = 2.0


diar_pipeline = Pipeline.from_pretrained(DIAR_MODEL, token=HF_TOKEN or None)
if DEVICE == "cuda":
    diar_pipeline.to(torch.device("cuda"))

with torch.inference_mode():
    diarization = diar_pipeline({"waveform": waveform, "sample_rate": sr})

raw_turns: list[tuple[float, float, str]] = []
label_to_id: dict[str, str] = {}
for turn, _, label_src in diarization.speaker_diarization.itertracks(yield_label=True):
    if label_src not in label_to_id:
        label_to_id[label_src] = f"S{len(label_to_id)}"
    raw_turns.append((turn.start, turn.end, label_to_id[label_src]))
raw_turns.sort(key=lambda t: t[0])


def merge_same_speaker(turns, max_gap_sec):
    if not turns:
        return []
    out = [turns[0]]
    for start, end, spk in turns[1:]:
        p_start, p_end, p_spk = out[-1]
        if spk == p_spk and start - p_end <= max_gap_sec:
            out[-1] = (p_start, max(p_end, end), p_spk)
        else:
            out.append((start, end, spk))
    return out


merged = merge_same_speaker(raw_turns, MERGE_GAP_SEC)

contract.speakers = [
    Speaker(id=spk_id, label_src=label_src)
    for label_src, spk_id in label_to_id.items()
]
contract.segments = [
    Segment(seg_id=f"seg_{i:04d}", start=round(s, 3), end=round(e, 3), speaker=spk)
    for i, (s, e, spk) in enumerate(merged)
]

print(f"Найдено спикеров: {len(contract.speakers)}")
print(f"Сегментов после склейки: {len(contract.segments)} (до склейки: {len(raw_turns)})\n")
n = 10
print(f"Первые {n} сегментов")
for seg in contract.segments[:n]:
    print(f"  [{seg.start} — {seg.end}]  {seg.speaker}")

/usr/local/lib/python3.12/dist-packages/pyannote/audio/models/blocks/pooling.py:103: UserWarning:

std(): degrees of freedom is <= 0. Correction should be strictly less than the reduction factor (input numel divided by output numel). (Triggered internally at /pytorch/aten/src/ATen/native/ReduceOps.cpp:1858.)



Найдено спикеров: 2
Сегментов после склейки: 40 (до склейки: 213)

Первые 10 сегментов
  [8.688 — 11.033]  S0
  [11.033 — 48.226]  S1
  [48.226 — 55.735]  S0
  [48.26 — 52.057]  S1
  [54.925 — 74.112]  S1
  [71.935 — 72.188]  S0
  [74.5 — 138.676]  S1
  [122.172 — 125.024]  S0
  [138.997 — 176.628]  S1
  [159.348 — 160.343]  S0


## 4. Транскрибация

Поулчаем пословные тайминги и используем для этого `faster-whisper large-v3`.

Распределяем слова по диар-сегментам: слово прикрепляется к сегменту, в чьи `[start, end]` попадает его центр.

Каждый `Segment` получает свой `text` и `words[]`, а его `start`/`end` подгоняются под границы реальных слов.

In [ ]:
%pip install -q --upgrade --no-cache-dir faster-whisper

In [ ]:
from faster_whisper import WhisperModel

ASR_MODEL_NAME = "large-v3"
LANGUAGE = "ru"

compute_type = "float16" if DEVICE == "cuda" else "int8"
asr_model = WhisperModel(ASR_MODEL_NAME, device=DEVICE, compute_type=compute_type)

asr_segments, info = asr_model.transcribe(
    str(AUDIO_PATH),
    language=LANGUAGE,
    beam_size=5,
    word_timestamps=True,
    vad_filter=True,
)

all_words: list[Word] = []
for s in asr_segments:
    for w in (s.words or []):
        token = (w.word or "").strip()
        if not token:
            continue
        all_words.append(Word(
            w=token,
            start=round(float(w.start), 3),
            end=round(float(w.end), 3),
            conf=round(float(w.probability), 4),
        ))

print(f"Whisper выдал {len(all_words)} слов. Язык: {info.language} (p={info.language_probability:.2f})\n")


buckets: dict[str, list[Word]] = {s.seg_id: [] for s in contract.segments}
seg_idx = 0
n = len(contract.segments)

for word in all_words:
    center = (word.start + word.end) / 2.0
    while seg_idx < n and contract.segments[seg_idx].end < center:
        seg_idx += 1
    if seg_idx >= n:
        break
    seg = contract.segments[seg_idx]
    if seg.start <= center <= seg.end:
        buckets[seg.seg_id].append(word)


# Заполняем каждый сегмент его словами и текстом.
for seg in contract.segments:
    seg_words = buckets[seg.seg_id]
    if not seg_words:
        seg.text = ""
        seg.words = []
        continue
    seg.text  = " ".join(w.w for w in seg_words).strip()
    seg.start = seg_words[0].start
    seg.end   = seg_words[-1].end
    seg.words = seg_words


for seg in contract.segments[:5]:
    print(f"  {seg.speaker} [{seg.start:6.2f} — {seg.end:6.2f}]: {seg.text}")

Whisper выдал 2636 слов. Язык: ru (p=1.00)

  S0 [  8.66 —  10.98]: Доброго здравия! Здравия желаем! По
  S1 [ 10.98 —  48.24]: -военному! Как звать? Представьтесь, пожалуйста. Перевозчиков Валерий Иванович, ветеран труда и войны. Руководитель военно -патриотического комплекса «Патриот» имени Героя Советского Союза летчика -штурмовика Пимена Григорьевича Колесникова. И послезавтра исполняется 35 лет торжественного открытия нашему ППК «Патриот». Судя по произношению, коренной белорус. — Ну, наполовину. — Но говоришь -то в Чите. — Родился в Забайкалье, в Чите. — Но говоришь -то по -белорусски. — Конечно, есть немножко что -то и от мамы белорусское, но отец -то уральский лесоруб, поэтому корни российские.
  S0 [ 48.24 —  55.34]: — Глубокая связь, вся Русь, вся Русь так оттуда. Хорошо, как ты здесь оказался? Расскажи, пожалуйста, с чем пришел, с чем пожил? — Вот это
  S1 [ 48.26 —  52.06]: 
  S1 [ 55.34 —  73.70]: мероприятие, которое мы называем «Левандия», пришедшее к нам из Ялты, оно по

## 5. Роли

Знаем, кто конкретно говорил (`S0`, `S1`), но не знаем, кто из них ведёт, а кто отвечает. Решим этот вопрос совокупностью 3 признаков:

1. Меньше суммарного времени речи — интервьюер задаёт короткие вопросы, а гость отвечает длинными ответами.
2. Короче средняя реплика — даже если интервьюер много говорит, его отдельные реплики короче.
3. Начинает первым — ведущий открывает интро.

In [ ]:
def pick_interviewer(speakers: list[Speaker], segments: list[Segment]) -> Optional[str]:
    stats: dict[str, dict[str, float]] = {}
    for spk in speakers:
        durs = [s.end - s.start for s in segments if s.speaker == spk.id]
        if not durs:
            continue
        stats[spk.id] = {
            "total":       sum(durs),
            "avg":         sum(durs) / len(durs),
            "first_start": min(s.start for s in segments if s.speaker == spk.id),
        }
    if len(stats) < 2:
        return None

    votes = {sid: 0 for sid in stats}
    votes[min(stats, key=lambda s: stats[s]["total"])]       += 1
    votes[min(stats, key=lambda s: stats[s]["avg"])]         += 1
    votes[min(stats, key=lambda s: stats[s]["first_start"])] += 1

    max_votes = max(votes.values())
    leaders = [sid for sid, v in votes.items() if v == max_votes]
    if len(leaders) == 1:
        return leaders[0]
    return min(leaders, key=lambda s: stats[s]["total"])


interviewer_id = pick_interviewer(contract.speakers, contract.segments)

for spk in contract.speakers:
    spk.role = "interviewer" if spk.id == interviewer_id else "respondent"

for spk in contract.speakers:
    total = sum(s.end - s.start for s in contract.segments if s.speaker == spk.id)
    print(f"  {spk.id} → {spk.role:12s}  (всего речи: {total:.1f} сек)")

  S0 → interviewer   (всего речи: 137.8 сек)
  S1 → respondent    (всего речи: 985.7 сек)


## 6. Сегментация на вопросы и ответы

Что имеем на данный момент:
- текст каждой реплики
- словные тайминги
- кто говорит
- кто из спикеров интервьюер.

Нам осталось понять, где в потоке вопросы и какие сегменты гостя на них отвечают.

Сделаем это следующим образом:

1. **Режем реплики интервьюера на предложения по знакам конца (`.`, `?`, `!`, `…`)** — конец предложения определяем по последнему символу токена (это возможно, потому что Whisper приклеивает пунктуацию к слову).

2. **Определяем вопросительные предложения по двум каналам:**
   - явный `?` в конце (Whisper может его пропустить);
   - первые 1-3 слова попадают в список вопросительных слов (`что`, `как`, `почему`, `расскажи`, ...) — это покрывает неявные вопросы без `?`.

3. **Схема пополнения вопроса такая:**
   - встретилось вопросительное предложение интервьюера - открываем новый `Question`;
   - встретился сегмент гостя - приписываем его как `answer_seg_ids` к текущему открытому вопросу;
   - встретили второе вопросительное предложение прежде хотя бы одного сегмент ответа - склеиваем в текущий вопрос (получается серия из 2-3 уточнений одной репликой).

In [ ]:
QUESTION_HINTS = frozenset({
    "что", "как", "где", "когда", "почему", "зачем", "кто", "сколько",
    "какой", "какая", "какое", "какие",
    "расскажи", "расскажите", "опиши", "опишите", "поделись", "поделитесь",
})
SENT_END  = (".", "?", "!", "…")
TOKEN_RE  = re.compile(r"^[^\wа-яёА-ЯЁ]+|[^\wа-яёА-ЯЁ]+$")  # очистка пунктуации с краёв токена


def split_sentences(seg: Segment) -> list[dict]:
    if not seg.words:
        text = (seg.text or "").strip()
        return [{"text": text, "start": seg.start, "end": seg.end}] if text else []

    sentences, cur = [], []
    for w in seg.words:
        cur.append(w)
        token = (w.w or "").rstrip()
        if token and token[-1] in SENT_END:
            sentences.append({
                "text":  " ".join(x.w for x in cur).strip(),
                "start": cur[0].start,
                "end":   cur[-1].end,
            })
            cur = []
    if cur:
        sentences.append({
            "text":  " ".join(x.w for x in cur).strip(),
            "start": cur[0].start,
            "end":   cur[-1].end,
        })
    return sentences


def is_question(text: str) -> bool:
    text = text.strip()
    if not text:
        return False
    if text.endswith("?"):
        return True
    for tok in text.split()[:3]:
        norm = TOKEN_RE.sub("", tok).lower()
        if norm and norm in QUESTION_HINTS:
            return True
    return False


questions: list[Question] = []
current: Optional[Question] = None
answered = False

for seg in contract.segments:
    if seg.speaker == interviewer_id:
        for sent in split_sentences(seg):
            if not is_question(sent["text"]):
                continue
            if current is not None and not answered:
                current.text = f"{current.text} {sent['text']}".strip()
                current.end  = sent["end"]
                continue
            if current is not None:
                questions.append(current)
            current = Question(
                start=sent["start"], end=sent["end"],
                text=sent["text"],
                asked_by=interviewer_id,
            )
            answered = False
    else:
        if current is not None:
            answered = True

if current is not None:
    questions.append(current)

contract.questions = questions

print(f"Найдено вопросов: {len(contract.questions)}\n")
for q in contract.questions:
    print(f"  [{q.start} — {q.end}]  {q.text}")

Найдено вопросов: 3

  [51.52 — 55.06]  Хорошо, как ты здесь оказался? Расскажи, пожалуйста, с чем пришел, с чем пожил?
  [540.56 — 567.26]  не то, что ты или я или кто -то другой. Понимания человека, что если ты хочешь мира, ты должен быть готов к войне. Как точно. Потому что никто сильного не трогает. Дергает того, кто вздыхает.
  [1028.28 — 1040.56]  Ну что ж, поговорили мы и про личное, мне тоже много воспоминаний нахлынуло, и про общественное, но в заключении нашим радиослушателям, пожелай что -то доброе, вечное, хорошее.


## 7. Эмоции в ответах гостя

Архитектурно устроен как замороженный `wav2vec2-base` (достаёт фичи звука) + обучаемая линейная голова на 5 эмоций
из датасета Dusha (`neutral / angry / positive / sad / other`). Применяем по скользящему окну
длиной 2 сек с шагом 1 сек по всему аудио

Голова обучается один раз и сохраняется в файл `ser_head_dusha.pt`, и при повторном запуске ноутбука, если есть веса - инференс, если весов нет - предлагается заново обучить голову, потом инференс

In [ ]:
import numpy as np
from transformers import Wav2Vec2Model, AutoFeatureExtractor
from datasets import load_dataset
from torch.utils.data import DataLoader
from tqdm.auto import tqdm

SER_BASE = "facebook/wav2vec2-base"
WEIGHTS_PATH = Path("ser_head_dusha.pt")     # ← файл, который сохраняется, чтобы при каждом запуске не переучивать голову
SER_LABELS   = ["neutral", "angry", "positive", "sad", "other"]
SER_HIDDEN   = 768          # размерность фич wav2vec2-base
SER_MAX_SEC  = 4            # длина окна при обучении


class SERHead(torch.nn.Module):
    def __init__(self, hidden_size: int, num_classes: int):
        super().__init__()
        self.classifier = torch.nn.Linear(hidden_size, num_classes)

    def forward(self, hidden_states):
        pooled = hidden_states.mean(dim=1)
        return self.classifier(pooled)


def train_head_on_dusha(
    base: Wav2Vec2Model,
    extractor: AutoFeatureExtractor,
    out_path: Path,
    train_size: int = 5_000,
    epochs: int = 3,
    batch_size: int = 32,
) -> SERHead:
    print(f"Загружаем nixiieee/dusha_balanced")
    ds  = load_dataset("nixiieee/dusha_balanced")
    sub = ds["train"].shuffle(seed=42).select(range(train_size))
    print(f"{len(sub)} примеров")

    SR, MAX_LEN = 16_000, 16_000 * SER_MAX_SEC

    def prepare(batch):
        arr = np.asarray(batch["audio"]["array"], dtype=np.float32)[:MAX_LEN]
        feats = extractor(arr, sampling_rate=SR,
                          max_length=MAX_LEN, truncation=True, padding="max_length",
                          return_tensors="np")
        return {"x": feats.input_values[0], "y": batch["emotion"]}

    proc = sub.map(prepare, remove_columns=sub.column_names, desc="preprocess")

    def collate(batch):
        x = torch.tensor(np.stack([b["x"] for b in batch]), dtype=torch.float32)
        y = torch.tensor([b["y"] for b in batch], dtype=torch.long)
        return x, y

    loader = DataLoader(proc, batch_size=batch_size, shuffle=True,
                        num_workers=2, pin_memory=True, collate_fn=collate)

    head = SERHead(SER_HIDDEN, num_classes=len(SER_LABELS)).to(DEVICE)
    base.eval()
    for p in base.parameters():
        p.requires_grad = False

    opt = torch.optim.AdamW(head.parameters(), lr=1e-3, weight_decay=1e-4)

    for epoch in range(epochs):
        pbar = tqdm(loader, desc=f"epoch {epoch+1}/{epochs}")
        running = 0.0
        for step, (x, y) in enumerate(pbar, 1):
            x = x.to(DEVICE, non_blocking=True)
            y = y.to(DEVICE, non_blocking=True)

            with torch.no_grad():
                hs = base(x).last_hidden_state

            opt.zero_grad(set_to_none=True)
            logits = head(hs)
            loss   = torch.nn.functional.cross_entropy(logits, y)
            loss.backward()
            opt.step()

            running += loss.item()
            if step % 20 == 0:
                pbar.set_postfix(loss=f"{running/step:.3f}")

    torch.save(head.state_dict(), out_path)
    print(f"Голова сохранена: {out_path.resolve()}")
    return head

In [ ]:
# Грузим базовую модель и экстрактор для неё
base  = Wav2Vec2Model.from_pretrained(SER_BASE).to(DEVICE)
extractor = AutoFeatureExtractor.from_pretrained(SER_BASE)
base.eval()
for p in base.parameters():
    p.requires_grad = False

# Если есть веса, то подгружаем, если нет, учим
head = SERHead(SER_HIDDEN, num_classes=len(SER_LABELS)).to(DEVICE)

if WEIGHTS_PATH.exists():
    print(f"Веса найдены: {WEIGHTS_PATH.resolve()} - пропускаем обучение")
    head.load_state_dict(torch.load(WEIGHTS_PATH, map_location=DEVICE))
else:
    agree_learn = input(f"Веса не найдены ({WEIGHTS_PATH}) - запустить обучение? (y/n)")
    if agree_learn == 'y':
        head = train_head_on_dusha(base, extractor, WEIGHTS_PATH)
    else:
        print("Пропускаем обучение (для успешного завершения работы проекта, обучение нужно будет пройти - перезапустите эту ячейку)")

head.eval()

# Инференс по текущему аудио.
WIN_SEC, STEP_SEC = 2.0, 1.0


def dominant_speaker(t_start: float, t_end: float) -> Optional[str]:
    """Кто говорит большую часть окна [t_start, t_end] по диаризации."""
    coverage: dict[str, float] = {}
    for seg in contract.segments:
        if seg.end <= t_start or seg.start >= t_end or seg.speaker is None:
            continue
        overlap = min(seg.end, t_end) - max(seg.start, t_start)
        if overlap > 0:
            coverage[seg.speaker] = coverage.get(seg.speaker, 0.0) + overlap
    return max(coverage, key=coverage.get) if coverage else None


@torch.no_grad()
def classify_window(t_start: float, t_end: float) -> dict[str, float]:
    """Прогон окна waveform через backbone + нашу голову."""
    samples = waveform[0, int(t_start * sr) : int(t_end * sr)].numpy()
    feats   = extractor(samples, sampling_rate=sr, return_tensors="pt").input_values.to(DEVICE)
    hs      = base(feats).last_hidden_state
    logits  = head(hs.float())
    probs   = logits.softmax(dim=-1)[0].cpu()
    return {SER_LABELS[i]: float(probs[i]) for i in range(len(SER_LABELS))}


emotion_timeline: list[EmotionPoint] = []
t = 0.0
while t + WIN_SEC <= duration_sec:
    spk = dominant_speaker(t, t + WIN_SEC)
    if spk is not None:  # окна с тишиной пропускаем
        scores = classify_window(t, t + WIN_SEC)
        emotion_timeline.append(EmotionPoint(
            start=round(t, 3),
            end=round(t + WIN_SEC, 3),
            speaker=spk,
            dominant=max(scores, key=scores.get),
            scores={k: round(v, 4) for k, v in scores.items()},
        ))
    t += STEP_SEC

contract.emotion_timeline = emotion_timeline

print(f"\nТочек на таймлайне эмоций: {len(emotion_timeline)}")
for p in emotion_timeline[:10]:
    print(f"  [{p.start:6.2f} — {p.end:6.2f}]  {p.speaker}  → {p.dominant}")

[transformers] Wav2Vec2Model LOAD REPORT from: facebook/wav2vec2-base
Key                          | Status     |  | 
-----------------------------+------------+--+-
project_hid.bias             | UNEXPECTED |  | 
project_hid.weight           | UNEXPECTED |  | 
quantizer.weight_proj.bias   | UNEXPECTED |  | 
quantizer.codevectors        | UNEXPECTED |  | 
project_q.bias               | UNEXPECTED |  | 
project_q.weight             | UNEXPECTED |  | 
quantizer.weight_proj.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Веса найдены: /content/ser_head_dusha.pt - пропускаем обучение

Точек на таймлайне эмоций: 1119
  [  7.00 —   9.00]  S0  → angry
  [  8.00 —  10.00]  S0  → positive
  [  9.00 —  11.00]  S0  → neutral
  [ 10.00 —  12.00]  S1  → neutral
  [ 11.00 —  13.00]  S1  → neutral
  [ 12.00 —  14.00]  S1  → neutral
  [ 13.00 —  15.00]  S1  → positive
  [ 14.00 —  16.00]  S1  → neutral
  [ 15.00 —  17.00]  S1  → neutral
  [ 16.00 —  18.00]  S1  → neutral


## 8. Финальный результат

`InterviewAnalysis` полностью заполнен. Сохраняем в JSON и строим визуализации

In [ ]:
OUTPUT_PATH = Path(f"{AUDIO_PATH.stem}.analysis.json")
OUTPUT_PATH.write_text(contract.model_dump_json(indent=2), encoding="utf-8")
print(f"Результат сохранён: {OUTPUT_PATH.resolve()}")


Визуализируем таймлайн эмоций

In [ ]:
import plotly.graph_objects as go
from IPython.display import Audio, display
import numpy as np

EMOTION_STYLE = {
    "neutral": ("Нейтральный", "#9aa0a6"),
    "positive": ("Радость", "#f5a623"),
    "sad": ("Грусть", "#4a90e2"),
    "angry": ("Злость", "#e74c3c"),
    "other": ("Другое", "#9b59b6"),
}

def _smooth(y: np.ndarray, win: int = 5) -> np.ndarray:
    if win <= 1 or len(y) < win:
        return y
    return np.convolve(y, np.ones(win) / win, mode="same")


def plot_emotion_timeline(contract, smooth_win: int = 5, show_speaker: str | None = None):
    # Выводим аудио сверху
    display(Audio(str(AUDIO_PATH)))

    points = contract.emotion_timeline
    if show_speaker:
        points = [p for p in points if p.speaker == show_speaker]
    if not points:
        print("Нет точек на таймлайне.")
        return

    t = np.array([(p.start + p.end) / 2 for p in points])

    fig = go.Figure()

    # Отрисовка самих линий эмоций
    for key, (label, color) in EMOTION_STYLE.items():
        y = np.array([p.scores.get(key, 0.0) * 100 for p in points])
        y = _smooth(y, smooth_win)
        fig.add_trace(go.Scatter(
            x=t, y=y, name=label,
            mode="lines",
            line=dict(color=color, width=2, shape="spline", smoothing=0.6),
            hovertemplate=f"{label}: %{{y:.1f}}%<extra></extra>",
        ))

    dur = contract.audio.duration_sec
    for i, q in enumerate(contract.questions):
        fig.add_vrect(
            x0=q.start, x1=q.end,
            fillcolor="rgba(255,255,255,0.05)",
            line=dict(color="#666", width=1, dash="dash"),
            layer="below",
        )
        answer_end = (contract.questions[i + 1].start
                      if i + 1 < len(contract.questions) else dur)
        if answer_end > q.end:
            fig.add_vrect(
                x0=q.end, x1=answer_end,
                fillcolor="rgba(245,166,35,0.07)",
                line=dict(color="#5a4220", width=1, dash="dot"),
                layer="below",
            )

    if contract.questions:
        qx     = [(q.start + q.end) / 2 for q in contract.questions]
        qlabel = [f"В.{i}" for i in range(1, len(contract.questions) + 1)]
        qhover = [
            f"<b>В.{i}</b>  [{q.start:.1f}—{q.end:.1f} с]<br>"
            f"<span style='font-size:11px'>{q.text}</span>"
            for i, q in enumerate(contract.questions, 1)
        ]
        fig.add_trace(go.Scatter(
            x=qx, y=[97] * len(qx),
            mode="markers+text",
            text=qlabel, textposition="top center",
            textfont=dict(color="#ddd", size=11),
            marker=dict(symbol="diamond", size=10, color="#bbb",
                        line=dict(color="#333", width=1)),
            hovertext=qhover, hoverinfo="text",
            name="Вопросы", showlegend=False,
        ))

    dur = int(contract.audio.duration_sec)
    step = 15 if dur < 600 else 60
    tickvals = list(range(0, dur + 1, step))
    ticktext = [f"{v // 60}:{v % 60:02d}" for v in tickvals]

    title = f"Таймлайн эмоций - {contract.audio.title}"
    if show_speaker:
        title += f"  ({show_speaker})"

    fig.update_layout(
        title=dict(text=title, font=dict(color="#eee", size=14)),
        template="plotly_dark",
        plot_bgcolor="#1e1e1e", paper_bgcolor="#1e1e1e",
        height=560,
        xaxis=dict(
            title="время",
            tickvals=tickvals, ticktext=ticktext,
            range=[0, dur],
            rangeslider=dict(visible=True, thickness=0.07, bgcolor="#2a2a2a"),
        ),
        yaxis=dict(title="вероятность", range=[0, 110], ticksuffix="%",
                   tickvals=[0, 25, 50, 75, 100]),
        hovermode="x unified",
        legend=dict(orientation="v", x=1.02, y=0.5,
                    bgcolor="rgba(0,0,0,0)", font=dict(color="#ddd")),
        margin=dict(l=60, r=160, t=60, b=80),
    )

    fig.show()


plot_emotion_timeline(contract)
# фильтр по одному спикеру:
# plot_emotion_timeline(contract, show_speaker="S1")

Так таймлайн выглядит на интервью `02_radio_interview_1900` (интерактивный график Plotly, на GitHub не отображается):

![Таймлайн эмоций](assets/emotion_timeline.png)


Визуализируем средние эмоции по каждому вопросу

In [ ]:
from IPython.display import HTML, display

CARD_COLORS = ["#7e57c2", "#2e8b57", "#e53935", "#fb8c00", "#1e88e5", "#00acc1"]


def _fmt_time(sec: float) -> str:
    sec = int(round(sec))
    return f"{sec // 60}:{sec % 60:02d}"


def render_question_cards(contract, top_n: int = 5):
    if not contract.questions:
      print("Нет вопросов для отрисовки.")
      return
    if not contract.emotion_timeline:
      print("emotion_timeline пуст.")
      return

    dur = contract.audio.duration_sec
    cards = []

    for i, q in enumerate(contract.questions):
        seg_start = q.start
        seg_end = (contract.questions[i + 1].start if i + 1 < len(contract.questions) else dur)

        # точки эмоций, чей центр окна попадает в сегмент
        seg_points = [p for p in contract.emotion_timeline
                      if seg_start <= (p.start + p.end) / 2 < seg_end]
        if not seg_points:
            continue
        # средние scores по сегменту → top_n по убыванию
        avg = {key: float(np.mean([p.scores.get(key, 0.0) for p in seg_points]))
               for key in EMOTION_STYLE}
        top = sorted(avg.items(), key=lambda kv: -kv[1])[:top_n]

        head_color = CARD_COLORS[i % len(CARD_COLORS)]
        time_range = f"{_fmt_time(seg_start)}–{_fmt_time(seg_end)}"

        rows = []
        for key, val in top:
            label, color = EMOTION_STYLE[key]
            pct = val * 100
            rows.append(f"""
            <div class="qrow">
              <div class="qmeta">
                <span class="qdot" style="color:{color}">●</span>
                <span class="qlabel">{label}</span>
                <span class="qval">{pct:.0f}%</span>
              </div>
              <div class="qbar"><div class="qfill"
                   style="width:{pct:.1f}%;background:{color}"></div></div>
            </div>""")

        cards.append(f"""
        <div class="qcard">
          <div class="qhead" style="background:{head_color}">
            <h3>Вопрос {i + 1}</h3>
            <span class="qtime">{time_range}</span>
          </div>
          <div class="qbody">{''.join(rows)}</div>
        </div>""")

    css = """
    <style>.qgrid { display:flex; gap:14px; flex-wrap:wrap;
               font-family:-apple-system,BlinkMacSystemFont,sans-serif;
               padding:8px 0; }
      .qcard { background:#2a2a2a; border-radius:10px; min-width:240px;
               flex:1 1 240px; overflow:hidden;
               box-shadow:0 2px 8px rgba(0,0,0,.35); }
      .qhead { padding:10px 14px; color:#fff;
               display:flex; justify-content:space-between; align-items:baseline; }
      .qhead h3 { margin:0; font-size:14px; font-weight:600; }
      .qtime  { font-size:11px; opacity:.85; }
      .qbody  { padding:14px; }
      .qrow   { margin-bottom:12px; }
      .qrow:last-child { margin-bottom:0; }
      .qmeta  { display:flex; align-items:baseline; font-size:12px;
                color:#ddd; margin-bottom:4px; }
      .qdot   { font-size:14px; margin-right:6px; line-height:1; }
      .qlabel { flex:1; }
      .qval   { color:#aaa; }
      .qbar   { background:#3a3a3a; height:5px;
                border-radius:3px; overflow:hidden; }
      .qfill  { height:100%; border-radius:3px; }
    </style>
    """

    display(HTML(css + f'<div class="qgrid">{"".join(cards)}</div>'))


render_question_cards(contract)

Средние эмоции по вопросам на том же интервью:

![Карточки вопросов](assets/question_cards.png)
